# Module 08 — Trajectory & Pseudotime Analysis

This notebook visualizes pre-computed trajectory and pseudotime results from
`scripts/08_trajectory.py`. PAGA (Partition-based Graph Abstraction) is used
to infer cell state transitions, and diffusion pseudotime (DPT) orders cells
along differentiation/degeneration trajectories.

**Key findings:**
- **NP compartment:** Pseudotime-condition correlation rho = -0.258 (degenerated
  cells at earlier pseudotime, suggesting dedifferentiation)
- **AF compartment:** rho = +0.341 (REVERSED — degenerated cells at later pseudotime)
- **CEP compartment:** rho = -0.163 (weak trend, similar to NP)
- AF reversal is a key finding warranting discussion

**Data sources:**
- `results/trajectories/paga_*.png` — PAGA connectivity graphs
- `results/trajectories/pseudotime_by_condition_*.png` — Pseudotime distributions
- `results/trajectories/pseudotime_by_celltype_*.png` — Cell type ordering
- `results/trajectories/gene_dynamics_*.png` — Gene expression over pseudotime
- `results/trajectories/pseudotime_correlations_*.tsv` — Statistical tests
- `results/trajectories/trajectory_genes_*.tsv` — Genes correlated with pseudotime

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = Path('..').resolve()
RESULTS = BASE / 'results' / 'trajectories'

print(f'Results directory: {RESULTS}')
print(f'PNG files: {len(list(RESULTS.glob("*.png")))}')
print(f'TSV files: {len(list(RESULTS.glob("*.tsv")))}')

## Pseudotime-Condition Correlations

Spearman correlation between pseudotime and condition (encoded as ordinal:
healthy < mild < severe). A negative correlation means degenerated cells
occupy earlier pseudotime positions; a positive correlation means degenerated
cells are at later positions.

In [ ]:
# Load and display correlation results for each compartment
compartments = ['NP', 'AF', 'CEP']

all_corrs = []
for comp in compartments:
    corr_path = RESULTS / f'pseudotime_correlations_{comp}.tsv'
    if corr_path.exists():
        df = pd.read_csv(corr_path, sep='\t')
        df.insert(0, 'compartment', comp)
        all_corrs.append(df)

if all_corrs:
    corr_df = pd.concat(all_corrs, ignore_index=True)
    display(corr_df.style.set_caption('Pseudotime-Condition Correlations by Compartment'))
else:
    print('No correlation files found')

## PAGA Connectivity Graphs

PAGA shows the connectivity between cell type clusters in the trajectory graph.
Edge thickness represents the strength of connection, revealing which cell
states are transcriptionally adjacent.

In [ ]:
for comp in compartments:
    paga_path = RESULTS / f'paga_{comp}.png'
    if paga_path.exists():
        display(Markdown(f'### {comp} — PAGA Graph'))
        display(Image(filename=str(paga_path), width=700))
    else:
        print(f'PAGA plot not found for {comp}')

## Pseudotime by Condition

Distribution of pseudotime values for each condition (healthy, mild degeneration,
severe degeneration). Shifts in the distribution indicate trajectory position
changes with disease.

In [ ]:
for comp in compartments:
    pt_cond_path = RESULTS / f'pseudotime_by_condition_{comp}.png'
    if pt_cond_path.exists():
        display(Markdown(f'### {comp} — Pseudotime by Condition'))
        display(Image(filename=str(pt_cond_path), width=700))
    else:
        print(f'Pseudotime by condition plot not found for {comp}')

## Pseudotime by Cell Type

Pseudotime ordering across cell types within each compartment. This reveals
the progression from one cell state to another along the trajectory.

In [ ]:
for comp in compartments:
    pt_ct_path = RESULTS / f'pseudotime_by_celltype_{comp}.png'
    if pt_ct_path.exists():
        display(Markdown(f'### {comp} — Pseudotime by Cell Type'))
        display(Image(filename=str(pt_ct_path), width=700))
    else:
        print(f'Pseudotime by cell type plot not found for {comp}')

## Gene Dynamics Along Pseudotime

Expression of key genes plotted as a function of pseudotime. These show how
marker gene expression changes along the trajectory, highlighting the genes
that drive cell state transitions.

In [ ]:
for comp in compartments:
    gene_path = RESULTS / f'gene_dynamics_{comp}.png'
    if gene_path.exists():
        display(Markdown(f'### {comp} — Gene Dynamics'))
        display(Image(filename=str(gene_path), width=800))
    else:
        print(f'Gene dynamics plot not found for {comp}')

## Trajectory-Associated Genes

Genes most strongly correlated with pseudotime in each compartment. These
are candidate drivers of the degeneration trajectory.

In [ ]:
for comp in compartments:
    tg_path = RESULTS / f'trajectory_genes_{comp}.tsv'
    if tg_path.exists():
        tg = pd.read_csv(tg_path, sep='\t')
        display(Markdown(f'### {comp} — Top Trajectory Genes'))
        print(f'Total trajectory genes: {len(tg)}')
        display(tg.head(20).style.set_caption(f'{comp}: Top 20 Trajectory-Associated Genes'))
    else:
        print(f'Trajectory genes file not found for {comp}')

## Overlap with Differential Expression

Trajectory-associated genes that also appear in the pseudobulk DE results
from Module 06. This overlap strengthens confidence in both analyses.

In [ ]:
for comp in compartments:
    overlap_path = RESULTS / f'trajectory_de_overlap_{comp}.tsv'
    if overlap_path.exists():
        overlap = pd.read_csv(overlap_path, sep='\t')
        display(Markdown(f'### {comp} — Trajectory-DE Overlap'))
        print(f'Overlapping genes: {len(overlap)}')
        if len(overlap) > 0:
            display(overlap.head(20).style.set_caption(
                f'{comp}: Genes in Both Trajectory and DE Analyses'))
    else:
        print(f'Trajectory-DE overlap file not found for {comp}')

## NP scVI-Based Trajectory (Alternative)

For the NP compartment, an alternative trajectory was computed using the scVI
latent space directly. This provides a complementary view to the standard
PCA-based trajectory.

In [ ]:
# NP scVI alternative plots
scvi_plots = [
    ('PAGA (scVI)', 'paga_NP_scVI.png'),
    ('Pseudotime by Condition (scVI)', 'pseudotime_by_condition_NP_scVI.png'),
    ('Pseudotime by Cell Type (scVI)', 'pseudotime_by_celltype_NP_scVI.png'),
]

for label, fname in scvi_plots:
    fpath = RESULTS / fname
    if fpath.exists():
        display(Markdown(f'**NP — {label}**'))
        display(Image(filename=str(fpath), width=700))

# scVI correlation
scvi_corr = RESULTS / 'pseudotime_correlations_NP_scVI.tsv'
if scvi_corr.exists():
    df = pd.read_csv(scvi_corr, sep='\t')
    display(Markdown('**NP scVI Pseudotime-Condition Correlation**'))
    display(df)

## Status — Module 08 Complete

### Summary
- PAGA + DPT trajectories computed for NP, AF, and CEP compartments
- Pseudotime-condition correlations reveal compartment-specific patterns
- Gene dynamics show key marker transitions along trajectories

### Key observations
| Compartment | Spearman rho | Interpretation |
|-------------|-------------|----------------|
| NP | -0.258 | Degenerated cells at earlier pseudotime (dedifferentiation) |
| AF | +0.341 | **REVERSED** — degenerated cells at later pseudotime |
| CEP | -0.163 | Weak trend, similar direction to NP |

### Discussion point: AF reversal
The positive correlation in AF (degenerated = later pseudotime) contrasts with
NP and CEP. Possible interpretations:
1. AF cells undergo a different degeneration program (fibrotic maturation rather
   than dedifferentiation)
2. AF trajectory captures a stress-induced differentiation response
3. Technical: AF trajectory direction may need reassessment

This finding requires careful interpretation and should be highlighted in the
manuscript as a compartment-specific difference.